# Evaluación de la traducción

1. Se tiene que hacer un filtrado y una limpieza por modelo. 
2. La limpieza automática se tiene que compensar con un filtrado manual, ya que los modelos NO SIEMPRE SIGUEN BIEN LAS INSTRUCCIONES.
3. Hay que medir los valores donde funcionan mal para reportarlo.

Valores interesantes que podemos analizar:

DeepSeek-R1-0528-Qwen3-8B
1. Validation 68-69, 179-180, 195-197.

GPT-OSS-20B
1. Validation 217, 218 (Para describir el etiqueado N/A).

In [1]:
import pandas as pd
import numpy as np
import re
import os
from nltk.inference.prover9 import *

### Filtrado Manual

In [2]:
def list_to_str(qwen_list):
    text = ''
    for value in qwen_list:
        new_val = re.sub('  ', '', value)
        text += new_val + '\n'
    return text

def get_trans(path, test):
    if test:
        start = 226 
    else:
        start = 203
    dataset = pd.read_csv(path)
    dataset = dataset.drop(columns = ["Unnamed: 0"])
    print("Longitud del dataset: {}".format(len(dataset)))
    trans = dataset[:start]
    return trans, path


dataset_path = [
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv', # DONE
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-Distill-Llama-8B_full.csv', # Está bien ojete cabrón. Ni para tratar de limpiarlo la neta. 
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-Distill-Qwen-1.5B_full.csv', # Está muy ojete. Tiene más sentido que arriba, pero nmms sigue siendo sida.
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-Distill-Qwen-7B_full.csv', # DONE 
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/gemma-3-4b-it_full.csv', # DONE
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/gemma-3-12b-it_full.csv', # DONE
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/gpt-oss-20b_full.csv', # DONE
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/Qwen3-4B-FP8_full.csv', # DONE (delicioso)
    '/home/flopezp/Ongoing/baseline_datasets/{}/raw/Qwen3-8B-FP8_full.csv', # DONE
    #'/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-14B-FP8_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-14B-FP8_full.csv', # DONE (delicioso as well)
    #'/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-4B_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-4B_full.csv', #DONE
    #'/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-9B_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-9B_full.csv'   
]

#folio_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
#folio_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)

folio_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
folio_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)

trans_test = get_trans(dataset_path[11].format('test'), True)
print("Longitudes de los subsplits:", len(trans_test[0]))

trans_val = get_trans(dataset_path[11].format('validation'), False)
print("Longitudes de los subsplits:", len(trans_val[0]))

Longitud del dataset: 678
Longitudes de los subsplits: 226
Longitud del dataset: 609
Longitudes de los subsplits: 203


In [3]:
def initial_filter(dataframe):
    df = dataframe[0]["Full"]
    filtrado = []

    for i in range(len(df)):
        if '</think>' in df[i]:
            think_split = wait_split.split('</think>')[-1]
            n_split = think_split.split('\n')
            aux_list = [value.split('::')[0] for value in n_split]
        else:
            dash_split = df[i].split('------')[0]
            inv_accent_split = dash_split.split('```')[0]
            wait_split = inv_accent_split.split('Wait, ')[0]
            n_split = wait_split.split('\n')
            aux_list = [value.split('::')[0] for value in n_split]

        for premise in aux_list:
            index = aux_list.index(premise)
            new_val = re.sub('(  )+', '', premise)
            aux_list[index] = new_val
        while '' in aux_list:
            aux_list.remove('')
        filtrado.append(aux_list)
    
    return filtrado
            

In [4]:
qwen35_9b_test = initial_filter(trans_test) # Al chile bastante bien para gemma y ds-distill, odiamos a gpt-oss, qwen3 (sexual)
qwen35_9b_val = initial_filter(trans_val)

def filter_sexual2(lista):
    aux_list = []
    for i in range(len(lista)):
        if "Natural Language Premises:" in lista[i]:
            index = lista[i].index("Natural Language Premises:")
            aux_list.append(lista[i][:index])
        else:
            aux_list.append(lista[i])
        
    for i in range(len(aux_list)):
        if "FOL Premises:" in aux_list[i]:
            index = aux_list[i].index("FOL Premises:")
            aux_list[i] = aux_list[i][index+1:]
        else:
            continue

    return aux_list

quasi_final = filter_sexual2(qwen35_9b_val)

In [5]:
# Filtrado para DeepSeek-R1-0528-Qwen3-8B

def clean_extra(instance):
   """
      Filtra las listas que tienen mucho texto además de las premisas.
   """
   try:
      for elem in instance:
         search = re.search("(FOL Premises)", elem)
         if search != None:
            val = instance.index(elem)
      new_list = instance[val+1:]
      return new_list
   except:
      return instance
   
def deepseek_clean(dataset):
    """
        Limpia el formato de salida de los modelos destilados de DeepSeek.
    """
    full_list = []
    for i in range(len(dataset)):
        text = dataset["Full"].iloc[i]
        aux = text.split('---')[0]
        split_vals = aux.split('\n')[:-1]
        for elem in split_vals:
            three_dots = re.search('(:::)', elem)
            if three_dots != None:
                new_val = elem.split(':::')[0]
                new_val = re.sub(r'(  )+|\t', '', new_val)
                split_vals[split_vals.index(elem)] = new_val
        full_list.append(split_vals)
    return full_list  

def deepseek_clean2(lista):
    """
        Segunda parte de la limpieza.
    """
    for elemento in lista:
        if len(elemento) > 10:
            index = lista.index(elemento)
            clean_list = clean_extra(elemento)
            lista[index] = clean_list
        if '' in elemento:
            while '' in elemento:
                elemento.remove('')
        for premise in elemento:
            index = elemento.index(premise)
            elemento[index] = re.sub('(  )+', '', premise)
    return lista

In [6]:
def print_weird_lists(dataset):
    print('-------------------------------')
    path = dataset[1]
    df = dataset[0]

    split_val = path.split('/')[-1]
    print("Quiubole con {}...".format(split_val[:-4]))

    aux = deepseek_clean(df)
    lista_limpia = deepseek_clean2(aux)

    for _ in lista_limpia:
        if len(_) > 10:
            index = lista_limpia.index(_)
            print(index, _)
    print('-------------------------------')

#print_weird_lists(trans_test)
#print_weird_lists(trans_val)

### LogicSim & Traducción

In [2]:
# LogicSim+
def get_nice_dict(regex, instance, const):
	"""
		A partir de una regex, extrae todos los valores encontrados y los formatea para tener un diccionario con apariciones.

		regex = str ; Expresión regular a buscar.
		const = bool ; Determina si estamos analizando constantes. Esto permite que se realice un filtro extra en el código.
	"""
	lista = []
	for _ in instance:
		aux = re.finditer(regex, _)
		for expression in aux:
			if const:
				regex_list = expression.group()[1:-1]
				regex_list = regex_list.split(',')
				for j in regex_list:
					if len(j) > 1:
						constant = re.sub(' ', '', j)
						lista.append(constant)
			else:
				lista.append(expression.group())

	set_set = list(set(lista))
	dict_dict = {}
	for _ in set_set:
		dict_dict["{}".format(_)] = lista.count(_)

	return dict_dict


def extract_info(ds_answer, llm):
	"""
		Limpia la respuesta de un dataset y extrae las premisas necesarias para calcular LogicSim.

		llm = Bool ; Señala si se va a evaluar la respuesta generada por un LLM.
	"""
	instance = ds_answer.split('\n')
	
    # Este bloque permite limpiar las respuestas que siguen en exceso el prompt inicial.
	if llm: 
		for i in range(len(instance)):
			instance[i] = re.sub('(::)+([ A-z.]+)', '', instance[i])
			instance[i] = re.sub('(:::)+([ A-z.]+)', '', instance[i])
			instance[i] = re.sub('(  )+', '', instance[i])

	pred_dict = get_nice_dict(r'[A-z]+\(([A-z]+(,? [A-z]+)*)\)', instance, False)
	const_dict = get_nice_dict(r'(\([A-z]+(\, [A-z]+){0,}\))', instance, True)
	logop_dict = get_nice_dict(r'[∀∧→⊕¬∨∃]+', instance, False)

	# len(instance) = Cantidad de premisas
	# pred_dict = Diccionario con cantidad de predicados. DE AQUÍ SE OBTIENE APARICIONES TOTALES Y CANTIDAD DE PREDICADOS
	# constant_dict =  Diccionario con cantidad de constantes. DE AQUÍ SE OBTIENE APARICIONES TOTALES Y CANTIDAD DE CONSTANTES
	# logop_dict =  Diccionario con cantidad de operadores y cuantificadores. DE AQUÍ SE OBTIENE APARICIONES TOTALES Y CANTIDAD DE CUANTIFICADORES
	return pred_dict, const_dict, logop_dict, len(instance)


def total_apps(dict1, preds):
    """
        Obtiene:
            1) La cantidad total de apariciones de un valor (Const/Pred/LogOps)
            2) La cantidad de constantes/predicados/logops distintos.

        dict1 = dict ; El diccionario obtenido previamente.
        preds = bool ; Valor booleano que permite realizar un filtro sobre los predicados.
    """
    if preds:
        lista_chida_aux = [re.search(r'([A-z]+\()', _).group() for _ in dict1.keys()]
        lista_chida = list(set(lista_chida_aux))

        # Guardamos la cantidad de apariciones en el diccionario previo.
        aux_dict = {"{}".format(_):0 for _ in lista_chida}
        for key in dict1.keys():
            for _ in lista_chida:
                if _ in key:
                    aux_dict["{}".format(_)] += dict1[key]
        
        if preds:
            aux_dict = {"{}".format(_[:-1]): aux_dict[_] for _ in aux_dict}
    else:
        aux_dict = dict1
    
    total_value = 0
    for _ in aux_dict:
        total_value += aux_dict[_]

    #total_value = Apariciones totales
    #len(aux_dict) = Cantidad de [VALUE] distinto.
    return total_value, len(aux_dict)


def logic_sim_plus_individual(llm_ans, folio_ans):
    """
        Extrae los valores distintos y valores totales de cada instancia. 
    """
    if llm_ans == 'nan':
        return llm_ans

    pred_llm, const_llm, logops_llm, len_llm = extract_info(llm_ans, True)
    pred_folio, const_folio, logops_folio, len_folio = extract_info(folio_ans, False)

    #llm
    dif_preds_llm, pred_count_llm = total_apps(pred_llm, True)
    dif_const_llm, const_count_llm = total_apps(const_llm, False)
    dif_logops_llm, logops_count_llm = total_apps(logops_llm, False)

    #folio
    dif_preds_folio, pred_count_folio = total_apps(pred_folio, True)
    dif_const_folio, const_count_folio = total_apps(const_folio, False)
    dif_logops_folio, logops_count_folio = total_apps(logops_folio, False)

    #Absolute Values
    dif_preds = abs(dif_preds_llm - dif_preds_folio)
    tot_aps_preds = abs(pred_count_llm - pred_count_folio)
    
    dif_const = abs(dif_const_llm - dif_const_folio)
    tot_aps_const = abs(const_count_llm - const_count_folio)

    dif_logops = abs(dif_logops_llm - dif_logops_folio)
    tot_aps_logops = abs(logops_count_llm - logops_count_folio)

    dif_premises = abs(len_llm - len_folio)

    logicsim_plus = dif_preds + tot_aps_preds + dif_const + tot_aps_const + dif_logops + tot_aps_logops + dif_premises
    return logicsim_plus

In [3]:
def clean_llm(dataset, value):
    """
        Filtra y deja bonito cada dataset para pasarlo por LogicSim+

        Sirve para DeepSeek-R1-0528, 
    """
    text_value = str(dataset["Translation"][value])
    llm_ans = re.sub('\', \'', r'\n', text_value)
    llm_ans = re.sub(r'[\'\"\[\].]', '', llm_ans)
    llm_ans = re.sub(' , ', r'\n', llm_ans)
    llm_ans = re.sub('  ', '', llm_ans)
    return llm_ans

In [4]:
dataset_path = [
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_DeepSeek-R1-Distill-Qwen-7B.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_gemma-3-4b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_gpt-oss-20b.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_Qwen3-4B-FP8.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_Qwen3-8B-FP8.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_Qwen3-14B-FP8.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_Qwen3.5-4B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_Qwen3.5-9B.csv'   
]

In [20]:
def get_logicsim_list(path, val):
    """
    Sirve para ver todas las salidas bonitas del conjunto de datos filtrado.
    """
    
    if val:
        path = path.format('validation')
        folio = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
    else:
        path = path.format('test')
        folio = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
    try:
        llm_ds = pd.read_csv(path)
        llm_ds = llm_ds.drop(columns = ["Unnamed: 0"])
    except:
        llm_ds = pd.read_csv(path)
    
    checkpoint_name = path.split('/')[-1][6:]
    print('================================================================')
    print('\t \t {}'.format(checkpoint_name))
    print('================================================================')
    

    logic_sim_list = []
    for i in range(len(llm_ds)):
        llm_instance = clean_llm(llm_ds, i)
        folio_instance = folio['premises-FOL'][i]
        logic_sim_list.append(logic_sim_plus_individual(llm_instance, folio_instance))
    
    return logic_sim_list

In [21]:
# Comparison between LLM vs FOLIO.
# Se puede hacer LLM vs LLM.

def average_logicsim(validation):
    if validation:
        print('---------------------')
        print('\t Validation')
        print('---------------------')
    else:
        print('---------------------')
        print('\t Test')
        print('---------------------')

    logsim_values = []
    for element in dataset_path:
        oal = get_logicsim_list(element, validation)
        logsim_values.append(oal)
        print("\t Cantidad total de NaN: {}".format(oal.count('nan')))
        while 'nan' in oal:
            oal.remove('nan')
        print("\t Avg:", round(np.mean(oal), 2), "Std:", round(np.std(oal), 2), "Var:", round(np.var(oal), 2))
    return logsim_values


validation_values = average_logicsim(True)
test_values = average_logicsim(False)

---------------------
	 Validation
---------------------
	 	 DeepSeek-R1-0528-Qwen3-8B.csv
	 Cantidad total de NaN: 0
	 Avg: 22.67 Std: 16.92 Var: 286.36
	 	 DeepSeek-R1-Distill-Qwen-7B.csv
	 Cantidad total de NaN: 13
	 Avg: 46.69 Std: 186.2 Var: 34670.73
	 	 gemma-3-4b-it.csv
	 Cantidad total de NaN: 0
	 Avg: 23.55 Std: 15.91 Var: 253.0
	 	 gemma-3-12b-it.csv
	 Cantidad total de NaN: 0
	 Avg: 18.79 Std: 15.18 Var: 230.3
	 	 gpt-oss-20b.csv
	 Cantidad total de NaN: 0
	 Avg: 25.6 Std: 21.54 Var: 464.13
	 	 Qwen3-4B-FP8.csv
	 Cantidad total de NaN: 0
	 Avg: 23.96 Std: 16.95 Var: 287.41
	 	 Qwen3-8B-FP8.csv
	 Cantidad total de NaN: 0
	 Avg: 22.0 Std: 16.49 Var: 272.03
	 	 Qwen3-14B-FP8.csv
	 Cantidad total de NaN: 0
	 Avg: 24.99 Std: 18.88 Var: 356.41
	 	 Qwen3.5-4B.csv
	 Cantidad total de NaN: 0
	 Avg: 25.12 Std: 16.64 Var: 277.0
	 	 Qwen3.5-9B.csv
	 Cantidad total de NaN: 2
	 Avg: 21.92 Std: 18.13 Var: 328.62
---------------------
	 Test
---------------------
	 	 DeepSeek-R1-0528-Qwen3-

In [ ]:
def show_2times_logic(oal_list):



# Revisión y expulsión de valores feos
aux = logic_sim_values[0]

high_count = [aux.index(value) if value > 45 else None for value in aux]
while None in high_count:
    high_count.remove(None)

# high_count corresponde a los índices cuyo valor asociado tiene un logicsim mayor a 2*avglogsim
quick = pd.read_csv('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv'.format('validation'))
quick = quick.drop(columns=["Unnamed: 0"])

for value in high_count:
    print(aux[value])
    print(folio['premises-FOL'][value])
    print('------')
    print(clean_llm(quick, value))
    print("============")

57
∀x (WildTurkey(x) → (EasternWildTurkey(x) ∨ OsceolaWildTurkey(x) ∨ GouldsWildTurkey(x) ∨ MerriamsWildTurkey(x) ∨ RiograndeWildTurkey(x) ∨ OcellatedWildTurkey(x)))
¬(EasternWildTurkey(tom))
¬(OsceolaWildTurkey(tom))
¬(GouldsWildTurkey(tom))
¬(MerriamsWildTurkey(tom) ∨ RiograndeWildTurkey(tom))
WildTurkey(tom)
------
There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, Gould’s wild turkey, Merriam’s wild turkey, Rio Grande wild turkey, and Ocellated wild turkey
Tom is not an Eastern wild turkey
Tom is not an Osceola wild turkey, Tom is not a Goulds wild turkey, Tom is neither a Merriams wild turkey nor a Rio Grande wild turkey, Tom is a wild turkey
57
∀x (WildTurkey(x) → (EasternWildTurkey(x) ∨ OsceolaWildTurkey(x) ∨ GouldsWildTurkey(x) ∨ MerriamsWildTurkey(x) ∨ RiograndeWildTurkey(x) ∨ OcellatedWildTurkey(x)))
¬(EasternWildTurkey(tom))
¬(OsceolaWildTurkey(tom))
¬(GouldsWildTurkey(tom))
¬(MerriamsWildTurkey(tom) ∨ RiograndeWildTurkey(tom))
WildTurkey(tom)
---

### Translation to Prover9

In [ ]:
def prove(argument):
    goal, assumptions = argument
    g = Expression.fromstring(goal)
    alist = [Expression.fromstring(a) for a in assumptions]
    p = Prover9Command(g, assumptions=alist).prove()
    return p

def get_full_ds(model):
    """
    model = str ; Debe de ser el nombre del LLM en cuestión: gpt, qwen, nemo, etc etc
    """
    ds1 = pd.read_csv(f'/home/flopezp/Kurosagol/Ongoing/datasets_new/{model}_trans.csv')
    ds2 = pd.read_csv(f'/home/flopezp/Kurosagol/Ongoing/datasets_new/{model}_infer.csv')
    ds3 = pd.read_csv(f'/home/flopezp/Kurosagol/Ongoing/datasets_new/{model}_retrans.csv')

    trans = ds1["Translation"]
    infer = ds2["Inference"]
    retrans = ds3["Retranslation"]

    full_ds = pd.DataFrame({"Translation": trans, "Inference": infer, "Retrans": retrans})
    return full_ds

# Amoxtli
full_qwen = get_full_ds('qwen')
full_gpt = get_full_ds('gpt')

full_qwen.head()

folio_full = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
folio_full.head()

#new_folio = load_dataset('fol-autoformalization/FOLIO', token = hf_token)
#new_folio

# Server Dra. Helena (bendiciones)
#tran_ds = pd.read_csv(r'/media/discoexterno/francisco/Kurosagol/Ongoing/datasets/gpt_qwen_translations_8ktokens.csv')
#tran_ds = tran_ds.drop(columns=["Unnamed: 0"])
#tran_ds.head()

,story_id,premises,premises-FOL,conclusion,conclusion-FOL,label,example_id
0,324,No professional soccer players play profession...,∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Pla...,Stephen Curry is an athlete.,Athlete(stephenCurry),Uncertain,828
1,324,No professional soccer players play profession...,∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Pla...,Stephen Curry is an athlete and a professional...,Athlete(stephenCurry) ∧ Professional(stephenCu...,False,829
2,324,No professional soccer players play profession...,∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Pla...,"If Stephen Curry is a professional centerback,...",(Professional(stephenCurry) ∧ CenterBack(steph...,True,830
3,357,If people join student government in high scho...,"∀x (Join(x, studentGovernment) → ManyOpportuni...",Tyler joined the student government in high sc...,"JoinIn(tyler, studentGovernment, highSchool)",Uncertain,947
4,357,If people join student government in high scho...,"∀x (Join(x, studentGovernment) → ManyOpportuni...",If Tyler either both stays alone in high schoo...,"¬(StayAloneIn(tyler, highSchool) ⊕ (Well-adjus...",True,948


In [ ]:
print("==================")
print("====始めましょう====")
print("==================")

gpt_vs_qwen = []
gpt_vs_folio = []
qwen_vs_folio = []

for i in range(len(full_qwen)):
    gpt = full_gpt["Translation"][i]
    qwen = full_qwen["Translation"][i]
    folio = folio_full["premises-FOL"][i]

    gptqwen = logic_sim_plus_individual(gpt, qwen)
    gptfolio = logic_sim_plus_individual(gpt, folio)
    qwenfolio = logic_sim_plus_individual(qwen, folio)
    print("GPT vs Qwen: {}".format(gptqwen))
    print("GPT vs FOLIO: {}".format(gptfolio))
    print("FOLIO vs Qwen: {}".format(qwenfolio))

    if gptqwen < 6:
        print("GPT vs QWEN < 6")
        print('-------')
        print(gpt)
        print('-------')
        print(qwen)
        print('-------')
        print(folio)
        print("================")
    
    if gptfolio > 25 or qwenfolio > 25:
        print("LLM vs FOLIO > 25")
        print('-------')
        print(gpt)
        print('-------')
        print(qwen)
        print('-------')
        print(folio)
        print("================")

    gpt_vs_qwen.append(gptqwen)
    gpt_vs_folio.append(gptfolio)
    qwen_vs_folio.append(qwenfolio)
    print('-------------------')

====始めましょう====
GPT vs Qwen: 2
GPT vs FOLIO: 16
FOLIO vs Qwen: 18
GPT vs QWEN < 6
-------
∀x (ProfessionalSoccerPlayer(x) → ¬ProfessionalBasketballPlayer(x))
∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x))
∀x (ProfessionalCenterBack(x) → ProfessionalSoccerDefender(x))
∃x (Athlete(x) ∧ ProfessionalCenterBack(x))
ProfessionalBasketballPlayer(stephenCurry)
-------


∀x (ProfessionalSoccerPlayer(x) → ¬PlaysBasketball(x))  
∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x))  
∀x (ProfessionalCenterback(x) → ProfessionalSoccerDefender(x))  
∃x (Athlete(x) ∧ ProfessionalCenterback(x))  
PlaysBasketball(StephenCurry)
-------
∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Play(x, professionalBasketball))
∀x ((Professional(x) ∧ Defender(x)) → (Professional(x) ∧ SoccerPlayer(x)))
∀x ((Professional(x) ∧ CenterBack(x)) → (Professional(x) ∧ Defender(x)))
∃x (Athlete(x) ∧ Professional(x) ∧ CenterBack(x))
Play(stephenCurry, professionalBasketball)
-------------------
GPT vs Q

In [ ]:
import statistics

print("LogicSim promedio (GPT vs Qwen): {}".format(round(sum(gpt_vs_qwen)/len(gpt_vs_qwen), 3)))
print("Desviación estándar: {}".format(round(statistics.stdev(gpt_vs_qwen), 3)))
print('----')
print("LogicSim promedio (GPT vs FOLIO): {}".format(round(sum(gpt_vs_folio)/len(gpt_vs_folio), 3)))
print("Desviación estándar: {}".format(round(statistics.stdev(gpt_vs_folio), 3)))
print('----')
print("LogicSim promedio (FOLIO vs Qwen): {}".format(round(sum(qwen_vs_folio)/len(qwen_vs_folio), 3)))
print("Desviación estándar: {}".format(round(statistics.stdev(qwen_vs_folio), 3)))

LogicSim promedio (GPT vs Qwen): 11.704
Desviación estándar: 13.428
----
LogicSim promedio (GPT vs FOLIO): 19.389
Desviación estándar: 13.347
----
LogicSim promedio (FOLIO vs Qwen): 23.279
Desviación estándar: 14.931


In [ ]:
# FOL to Prover9 Syntax

def switch_quantifiers(text, cuantifier):
    """
        Elimina todos los cuantificadores y los traduce a sintaxis de Prover9. Sin importar la variable ni la cantidad de apariciones. Qué pedo soy una verga para esto.

        text = str ;  Texto a modificar.
        cuantifier = str ('forall', 'exists') ; Cuantificador a modificar.
    """
    if cuantifier == 'forall':
        regex = '∀[A-z]'
        cuant = 'all '
    else:
        regex = '∃[A-z]'
        cuant = 'exists '

    owo = re.finditer(regex, text)
    aux_list = list(owo)
    if len(aux_list) == 0:
        return text

    # Redefinimos el iterador porque hacer lista de un iterador lo consume. CHINGA TU MADRE PYTHON. VETE A LA BURGER.    
    owo = re.finditer(regex, text)
    temporal_str = ''

    for _ in owo:
        if temporal_str == '':
            temporal_str = text[0:_.start()] + cuant + _.group()[-1] + '.' + text[_.end():]
        else:
            value = re.search(regex, temporal_str)
            temporal_str = temporal_str[0:(value.start())] + cuant + value.group()[-1] + '.' + temporal_str[value.end():]
    
    return temporal_str



def fol_to_prover9(value):
    """
        Modifica los símbolos lógicos normales y los cambia por los valores adecuados para Prover9.*
    """
    temp = value.lower()
    temp = switch_quantifiers(temp, 'forall')
    temp = switch_quantifiers(temp, 'exists')
    temp = re.sub('-', '', temp)
    temp = re.sub('¬', '-', temp) 
    temp = re.sub('→|→', '-> ', temp)
    temp = re.sub('∧', '&', temp)
    temp = re.sub('∨', '|', temp)
    temp = re.sub('↔', '<->', temp)
    temp = re.sub('≠', '!=', temp)
    temp = re.sub(r'\'', '', temp)
    temp = re.sub(r'\[', '(', temp)
    temp = re.sub(r'\]', ')', temp)
    temp = re.sub('∴', '', temp)
    temp = re.sub('(\. \()', '.(', temp)
    temp = re.sub('\.{2}', '.', temp)
    temp = re.sub(r'\)\.', ')', temp)
    temp = re.sub(r'([^a-z]\.\()', '(', temp)
    return temp


def dash_predicates(text):
    """
        Cambia los predicados de la forma "texto-texto-texto(x)" -> "textotextotexto(x)"

        text = str ; el hilo a modificar.
    """
    all_values = len(re.findall(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', text))

    if all_values == 0:
        return text
    
    new_text = text
    for i in range(all_values):
        current_regex = re.search(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


def elim_spaces(text):
    """
        Elimina los espacios entre variables: lionel messi -> lionelmessi

        text = str; El texto a modificar.


        OBS: Este formato de funciones (Encontrar cantidades y luego iterar sobre las cantidades) me gusta bastante.
    """
    total_iters = len(list(re.finditer('([A-z]+ )+([A-z]{2,})', text)))
    if total_iters == 0:
        return text

    new_text = text
    for i in range(total_iters):
        current_regex = re.search('([A-z]+ )+([A-z]{2,})', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


# El XOR me tiene hasta los huevos cabrón te lo juro.
def xor_bonito(expression):
    """
        Elimina el símbolo de XOR, y lo reescribe en la fórmula (A OR B) AND NOT(A AND B)
    """
    individual_values = re.findall('([A-z|_|0-9]{2,}|¬)', expression)
    a = individual_values[0] + '(x)'
    b = individual_values[1] + '(x)'
    a_or_b = '(' + a + ' | ' + b + ')'
    not_a_and_b = '-(' + a + ' & ' + b +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def xor_bonito_extreme(expression):
    """
        Elimina los XOR de fórmulas compuestas.
    """
    aux2 = re.search(r'([a-z]+\([a-z, ]+\)) ⊕ ([a-z]+\([a-z, ]+\))', expression)
    split = aux2.group().split('⊕')
    a_f = split[0]
    b_f = split[-1]
    a_or_b = '(' + a_f + ' | ' + b_f + ')'
    not_a_and_b = '-(' + a_f + ' & ' + b_f +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def rewrite_xor(re_search, element, comp):
    """
        Genera una nueva expresión a partir del xor bonito. 

        re_search = re.search(regex, str)
        element = str ; same str as above
        comp = bool ; True iff predicates have multiple variables.
    """
    start = re_search.start()
    end = re_search.end()

    xor_substr = element[start:end]
    if comp:
        xor_chido = xor_bonito_extreme(xor_substr)
    else:
        xor_chido = xor_bonito(xor_substr)

    nuevo = element[:start] + xor_chido + element[end:]
    return nuevo


def clean(value):
    """
        Procesa una respuesta individual de FOLIO/GPT_TRANS/QWEN_TRANS para que se pase al formato de Prover9.
    """

    clean_premises_aux = value.split('\n')
    clean_premises = []
    for _ in clean_premises_aux:
        if _ != '':
            clean_premises.append(_)

    if "Premises" in clean_premises[0]:
        del clean_premises[0]

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub('(:::)+([ A-z.]+)', '', _)

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub('[0-9]\.', '', _)

    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = fol_to_prover9(instance)

    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = elim_spaces(instance)

    try:
        # Filtro XOR sencillo
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                while xor_count > 0:
                    aux = re.search('(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element)
                    element = rewrite_xor(aux, element, False)
                    xor_count = len(re.findall('⊕', element))
                clean_premises[clean_premises.index(value)] = element

        # Filtro XOR multipremisa
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                while xor_count > 0:
                    aux = re.search('(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element)
                    element = rewrite_xor(aux, element, True)
                    xor_count = len(re.findall('⊕', element))
                clean_premises[clean_premises.index(value)] = element

    except:
        print("Hubo un Xor malo")    

    return clean_premises

<>:55: SyntaxWarning: invalid escape sequence '\.'
<>:56: SyntaxWarning: invalid escape sequence '\.'
<>:175: SyntaxWarning: invalid escape sequence '\.'
<>:190: SyntaxWarning: invalid escape sequence '\('
<>:201: SyntaxWarning: invalid escape sequence '\('
<>:55: SyntaxWarning: invalid escape sequence '\.'
<>:56: SyntaxWarning: invalid escape sequence '\.'
<>:175: SyntaxWarning: invalid escape sequence '\.'
<>:190: SyntaxWarning: invalid escape sequence '\('
<>:201: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_63037/989746200.py:55: SyntaxWarning: invalid escape sequence '\.'
  temp = re.sub('(\. \()', '.(', temp)
/tmp/ipykernel_63037/989746200.py:56: SyntaxWarning: invalid escape sequence '\.'
  temp = re.sub('\.{2}', '.', temp)
/tmp/ipykernel_63037/989746200.py:175: SyntaxWarning: invalid escape sequence '\.'
  clean_premises[clean_premises.index(_)] = re.sub('[0-9]\.', '', _)
/tmp/ipykernel_63037/989746200.py:190: SyntaxWarning: invalid escape sequence '\('
  aux = re

In [ ]:
for i in range(len(full_qwen)):
    print("=============================")
    print("============={}==============".format(i))
    print("=============================")
    print(folio_full['premises-FOL'][i])
    print('-----------------')
    print(clean(folio_full['premises-FOL'][i]))
    print(clean(full_qwen['Translation'][i]))
    print(clean(full_gpt['Translation'][i]))

=============0==============
∀x ((Professional(x) ∧ SoccerPlayer(x)) → ¬Play(x, professionalBasketball))
∀x ((Professional(x) ∧ Defender(x)) → (Professional(x) ∧ SoccerPlayer(x)))
∀x ((Professional(x) ∧ CenterBack(x)) → (Professional(x) ∧ Defender(x)))
∃x (Athlete(x) ∧ Professional(x) ∧ CenterBack(x))
Play(stephenCurry, professionalBasketball)
-----------------
['all x.((professional(x) & soccerplayer(x)) ->  -play(x, professionalbasketball))', 'all x.((professional(x) & defender(x)) ->  (professional(x) & soccerplayer(x)))', 'all x.((professional(x) & centerback(x)) ->  (professional(x) & defender(x)))', 'exists x.(athlete(x) & professional(x) & centerback(x))', 'play(stephencurry, professionalbasketball)']
['all x.(professionalsoccerplayer(x) ->  -playsbasketball(x))  ', 'all x.(professionalsoccerdefender(x) ->  professionalsoccerplayer(x))  ', 'all x.(professionalcenterback(x) ->  professionalsoccerdefender(x))  ', 'exists x.(athlete(x) & professionalcenterback(x))  ', 'playsbasketb

In [ ]:
def check_arg_validity(index, llm):
    """
        Dadas premisas en lenguaje lógico, verifica que la conclusión correspondiente sea inferible.
    """
    llm_valid, llm_invalid, fol_valid, fol_invalid = 0, 0, 0, 0
    if llm == 'qwen':
        llm_ds = full_qwen
    else:
        llm_ds = full_gpt
    llm_value = llm_ds['Translation'.format(llm)][index]
    folio_value = folio_full['premises-FOL'][index]

    true_validity = folio_full['label'][index]
    print("Logical Validity: {}".format(true_validity))

    clean_llm = clean(llm_value)
    clean_folio = clean(folio_value)
    clean_conc = clean(folio_full['conclusion-FOL'][index])
    
    args_llm = (
        clean_conc[0],
        clean_llm
    )

    args_folio = (
        clean_conc[0],
        clean_folio
    )
    
    # Lo que hacemos aquí es evaluar una conclusión contra las premisas de un LLM o del conjunto de datos.
    # OBS: A veces la premisa NO debe de ser deducible. 
    # ¿Cómo vamos a cuantificar este show?
    # Casos:
    # Premisas: TRUE, UNCERTAIN, FALSE.
    # Posibles resultados: TRUE, FALSE. ¿Cómo lidiamos con Uncertain? -> En estos casos Prover9 te regresa False. 
    
    print("LLM ({}):".format(llm))
    try:
        print("Ejecutable. Valor: {}".format(prove(args_llm)))
        llm_valid += 1
    except Exception as e:
        print(e)
        print(type(e))
        llm_invalid += 1
        
    print("FOLIO:")
    try:
        print("Ejecutable. Valor: {}".format(prove(args_folio)))
        fol_valid += 1
    except Exception as e:
        print(e)
        print(type(e))
        fol_invalid += 1
    print("======================")

    return llm_valid, llm_invalid, fol_valid, fol_invalid
    

**ACHTUNG** ESTO SE TIENE QUE CORRER EN EL CLUSTER DE HELENAAAAAAAAAAAAAAAAAAAAAA QUE NECESITAMOS A PROVER9 JIJIJIJUJUJUJU XD

In [ ]:
_llm_valid, _llm_invalid, _fol_valid, _fol_invalid = 0,0,0,0
for i in range(len(full_qwen)):
    print(i)
    try:
        a, b, c, d = check_arg_validity(i, 'qwen')
        _llm_valid += a
        _llm_invalid += b
        _fol_valid += c
        _fol_invalid += d
    except:
        print("oops")

print("Premisas de LLM válidas: {}".format(_llm_valid))
print("Premisas de LLM inválidas: {}".format(_llm_invalid))
print("Premisas de FOLIO válidas: {}".format(_fol_valid))
print("Premisas de FOLIO inválidas: {}".format(_fol_invalid))


0
Logical Validity: Uncertain
LLM (qwen):


NLTK was unable to find the prover9 file!
Use software specific configuration parameters or set the PROVER9 environment variable.

  Searched in:
    - /usr/local/bin/prover9
    - /usr/local/bin/prover9/bin
    - /usr/local/bin
    - /usr/bin
    - /usr/local/prover9
    - /usr/local/share/prover9

  For more information on prover9, see:
    <https://www.cs.unm.edu/~mccune/prover9/>
<class 'LookupError'>
FOLIO:


NLTK was unable to find the prover9 file!
Use software specific configuration parameters or set the PROVER9 environment variable.

  Searched in:
    - /usr/local/bin/prover9
    - /usr/local/bin/prover9/bin
    - /usr/local/bin
    - /usr/bin
    - /usr/local/prover9
    - /usr/local/share/prover9

  For more information on prover9, see:
    <https://www.cs.unm.edu/~mccune/prover9/>
<class 'LookupError'>
1
Logical Validity: False
LLM (qwen):


NLTK was unable to find the prover9 file!
Use software specific configuration parameters 

In [ ]:
# Vamos a tomarnos una pausa de la traducción a la sintaxis de Prover9.
# Todavía falta por ver unos casos tipo: e(x) y mejorar el hdspm del XOR.
# Ahora vamos a generar las respuestas del modelo para el proceso de inferencia.